In [1]:
# Cell 1 — Imports
from pathlib import Path
import sys

PROJECT_ROOT = Path(".").resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

from utils.io_utils import load_config, load_model
from gcamp_analysis.experiments.tree import ExperimentTreeBuilder, is_video_dir, print_tree
from gcamp_analysis.video_runner import VideoPipelineRunner
from gcamp_analysis.experiments.processor import ExperimentProcessor
from gcamp_analysis.experiments.io import save_comparisons, save_treatment_comparisons

In [2]:
config_path = PROJECT_ROOT / "config" / "notebook_config.yaml"
config = load_config(config_path)

print(f"Config: {config_path}")

Config: C:\Users\mzinn1\Desktop\Scripts\GCaMP-analysis\config\notebook_config.yaml


In [3]:
roi_model, roi_cfg = load_model(config["models"], which="roi")
spike_model, spike_cfg = load_model(config["models"], which="spike")

models = {
    "roi": roi_model,
    "roi_config": roi_cfg,
    "spike": spike_model,
    "spike_config": spike_cfg,
}

runner = VideoPipelineRunner.build(config, models)

print(f"ROI model:   {type(roi_model).__name__}")
print(f"Spike model: {type(spike_model).__name__}")

ROI model:   RandomForestClassifier
Spike model: LogisticRegression


In [4]:
EXPERIMENT_ROOT = Path(r"C:\Users\mzinn1\Desktop\invivo_tiffs")  # TODO: change per experiment
assert EXPERIMENT_ROOT.exists(), f"Experiment root not found: {EXPERIMENT_ROOT}"

builder = ExperimentTreeBuilder(is_video_dir=is_video_dir)
tree = builder.build(EXPERIMENT_ROOT)
print_tree(tree)

└── invivo_tiffs
    ├── 5729L-10
    ├── 5729L-12
    ├── 5729L-13
    ├── 5729L-14
    ├── 5729L-15
    ├── 5729L-16
    ├── 5729L-18
    ├── 5729L-19
    ├── 5729L-21
    ├── 5729L-26
    ├── 5729L-27
    ├── 5729L-29
    ├── 5729L-8
    ├── 5729R-2
    ├── 5729R-5
    ├── 5729R-7
    ├── 5729R-8
    ├── 5730R-10
    ├── 5730R-12
    ├── 5730R-14
    ├── 5730R-15
    ├── 5730R-16
    ├── 5730R-4
    ├── 5730R-5
    ├── 5730R-6
    ├── 5730R-7
    ├── 5730R-8
    ├── 5732L-2
    ├── 5732L-5
    ├── 5732L-7
    ├── 5732L-8
    ├── 5732R-10
    ├── 5732R-2
    ├── 5732R-4
    ├── 5732R-6
    ├── 5732R-7
    ├── 5732R-9
    ├── 5735L-10
    ├── 5735L-3
    ├── 5735L-4
    ├── 5735L-8
    ├── 5735L-9
    ├── 5735R-1
    ├── 5735R-10
    ├── 5735R-11
    ├── 5735R-12
    ├── 5735R-13
    ├── 5735R-14
    ├── 5735R-15
    ├── 5735R-16
    ├── 5735R-17
    ├── 5735R-2
    ├── 5735R-4
    ├── 5735R-5
    ├── 5735R-6
    ├── 5735R-9
    └── metrics


In [5]:
processor = ExperimentProcessor(
    runner=runner,
    output_root=EXPERIMENT_ROOT,
)
processor.process_tree(tree, verbose=True)


 Processing: 5729L-10
  Traces: 23 ROIs, 200 frames @ 3.0 Hz
  ROI filter: 1/23 kept (4.3%)
  Spikes: 3/17 kept | neurons 1 → 1
  <2 neurons with spikes — skipping grouping.

 Processing: 5729L-12
  Traces: 40 ROIs, 200 frames @ 3.0 Hz
  ROI filter: 3/40 kept (7.5%)
  Spikes: 6/83 kept | neurons 3 → 3
  Grouping (light-evoked): | light-evoked=1

 Processing: 5729L-13
  Traces: 47 ROIs, 200 frames @ 3.0 Hz
  ROI filter: 6/47 kept (12.8%)
  Spikes: 20/148 kept | neurons 6 → 6
  Grouping (light-evoked): | light-evoked=3

 Processing: 5729L-14
  Traces: 24 ROIs, 200 frames @ 3.0 Hz
  ROI filter: 5/24 kept (20.8%)
  Spikes: 23/90 kept | neurons 5 → 5
  Grouping (light-evoked): | light-evoked=3

 Processing: 5729L-15
  Traces: 21 ROIs, 200 frames @ 3.0 Hz
  ROI filter: 4/21 kept (19.0%)
  Spikes: 18/99 kept | neurons 4 → 4
  Grouping (light-evoked): | light-evoked=3

 Processing: 5729L-16
  Traces: 25 ROIs, 200 frames @ 3.0 Hz
  ROI filter: 4/25 kept (16.0%)
  Spikes: 22/97 kept | neurons 4

In [6]:
 
sibling_tables = processor.compare_siblings(tree)

for node_path, df in sibling_tables.items():
    if len(df) >= 2:
        print(f"\nNode: {node_path}")
        print(df.to_string(index=False))


Node: C:\Users\mzinn1\Desktop\invivo_tiffs
   child  n_videos  n_neurons  frac_grouped  frac_ungrouped  n_groups_light-evoked  mean_group_size_light-evoked  median_group_size_light-evoked  mean_group_corr_light-evoked  mean_spikes_per_group_light-evoked  decay_tau_seconds_mean_unweighted  half_max_width_seconds_mean_unweighted  rise_slope_hz_mean_unweighted  decay_tau_seconds_mean_weighted  half_max_width_seconds_mean_weighted  rise_slope_hz_mean_weighted  decay_tau_seconds_mean_ungrouped  half_max_width_seconds_mean_ungrouped  rise_slope_hz_mean_ungrouped  spike_frequency_mean_unweighted  spike_frequency_mean_weighted  spike_frequency_mean_ungrouped  decay_tau_seconds_mean_grouped  rise_slope_hz_mean_grouped  spike_frequency_mean_grouped  half_max_width_seconds_mean_grouped  decay_tau_seconds_var_unweighted  decay_tau_seconds_within_unweighted  decay_tau_seconds_between_unweighted  half_max_width_seconds_var_unweighted  half_max_width_seconds_within_unweighted  half_max_width_seconds

In [7]:
save_comparisons(
    root=tree,
    sibling_tables=sibling_tables,
    output_subdir="metrics",
    filename="sibling_comparisons.xlsx",
)

save_treatment_comparisons(tree)